# DSPy extract

In [8]:
model_config_file = "../output/models/extractor_config.json"
output_dir = "../output/dspy-extract/"
input_dir = "../input/pkna"

In [3]:
from typing import Literal

import dspy
from pydantic import BaseModel, Field


class UnoDetection(dspy.Signature):
    """Detect the presence of the character 'Uno' in a comic book page.

    Description of the character: has a duck-like appearance, inside a sphere that appears to be made of a bright green gelatinous substance, with small bubbles. It has a short, rounded beak, large, black eyes without defined pupils."""

    page: dspy.Image = dspy.InputField(desc="Comic book page image")
    uno_present: bool = dspy.OutputField(
        desc="Is the character 'Uno' present in the page?"
    )


Character = Literal['uno', 'pk', 'other']


class ExtractedLine(BaseModel):
    """A line of dialogue extracted from a comic book page."""
    character: Character = Field(
        description="The character who said the line."
    )
    text: str = Field(
        description="The text of the dialogue line."
    )


class DialogueExtraction(dspy.Signature):
    """Extract dialogues from a comic book page.

    The dialogues are expected to be in the form of text bubbles, typically found in comic books."""

    page: dspy.Image = dspy.InputField(desc="Comic book page image")
    dialogue: list[ExtractedLine] = dspy.OutputField(
        desc="List of dialogues extracted from the page, each with a character name and text"
    )


class ComicBookPage(BaseModel):
    """A class representing a comic book page."""
    uno_present: bool = False
    dialogue: list[ExtractedLine] = Field(default_factory=list)


class ComicBookExtractor(dspy.Module):

    def __init__(self):
        self.detect_uno = dspy.ChainOfThought(UnoDetection)
        self.extractor = dspy.ChainOfThought(DialogueExtraction)
        self.normalize = dspy.Predict(
            dspy.make_signature(
                signature='text -> normalized',
                instructions="Normalize text by using normal caps instead of all caps, and accented letters instead of apostrophes when appropriate.",
            )
        )

    def forward(self, img: dspy.Image) -> ComicBookPage:
        if not self.detect_uno(page=img).uno_present:
            return ComicBookPage(uno_present=False)

        extracted = self.extractor(page=img).dialogue
        normalized = [
            self.normalize(text=line.text).normalized
            for line in extracted
        ]
        return ComicBookPage(
            uno_present=True,
            dialogue=[
                ExtractedLine(character=line.character, text=normalized_text)
                for line, normalized_text in zip(extracted, normalized)
            ],
        )
        

In [21]:
import os
from dotenv import load_dotenv

load_dotenv()

lm = dspy.LM(
    model='vertex_ai/gemini-2.5-flash',
    vertex_credentials=os.getenv('VERTEX_AI_CREDS'),
    temperature=1.0,
    max_tokens=65535,
)
dspy.configure(lm=lm)
#dspy.settings.configure(track_usage=True)

extractor = ComicBookExtractor()
extractor.load(model_config_file)

In [18]:
from dataclasses import dataclass

@dataclass
class ExtractedPage:
    input_path: str
    extracted: ComicBookPage

    def to_dict(self):
        return {
            'input_path': self.input_path,
            'extracted': {
                'uno_present': self.extracted.uno_present,
                'dialogue': [
                    {
                        'character': line.character,
                        'text': line.text
                    } for line in self.extracted.dialogue
                ]
            },
        }

In [19]:
import json

def extract_from_page(input_path: str, output_path: str) -> None:
    if os.path.exists(output_path):
        print(f"Output file {output_path} already exists, skipping extraction.")
        return

    img = dspy.Image.from_file(input_path)
    extracted = extractor(img)
    extracted_page = ExtractedPage(input_path=input_path, extracted=extracted)
    with open(output_path, 'w') as f:
        json.dump(extracted_page.to_dict(), f, ensure_ascii=False, indent=2)

In [22]:
import os
from glob import glob
import tqdm

# Prepare the output directory
os.makedirs(output_dir, exist_ok=True)

# Process each directory in the input directory
input_dirs = sorted(glob(os.path.join(input_dir, '*')))

for pkna_dir in input_dirs:
    print(f"Processing directory: {pkna_dir}")
    output_path = os.path.join(output_dir, os.path.basename(pkna_dir))
    os.makedirs(output_path, exist_ok=True)
    pages = sorted(
        glob(f'{pkna_dir}/*.jpg') + glob(f'{pkna_dir}/*.jpeg')
    )

    for page_path in tqdm.tqdm(pages, desc="Processing pages", unit="page"):
        print(f"Processing page: {page_path}")
        output_file = os.path.join(output_path, f"{os.path.splitext(os.path.basename(page_path))[0]}.json")
        extract_from_page(page_path, output_file)

Processing directory: ../input/pkna/pkna-0


Processing pages:   0%|          | 0/71 [00:00<?, ?page/s]

Processing page: ../input/pkna/pkna-0/pkna-0-000.jpg


Processing pages:   1%|▏         | 1/71 [00:08<10:25,  8.94s/page]

Processing page: ../input/pkna/pkna-0/pkna-0-001.jpg


...

Processing page: ../input/pkna/pkna-0/pkna-0-002.jpg


Processing pages:   4%|▍         | 3/71 [00:24<09:12,  8.12s/page]

Processing page: ../input/pkna/pkna-0/pkna-0-003.jpg


Processing pages:   4%|▍         | 3/71 [00:25<09:31,  8.40s/page]


KeyboardInterrupt: 